##Creating a table of relevant MSDS variables linked to CCU063 maternity interpreter cohort 

Purpose - to link MSDS gestational age at booking with maternity interpreter cohort

Authors - Majel McGranahan supported by Lars Murdock

Reviewed - Not reviewed, needs cleaning up by MM

#0 Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

# 1 Load Maternity Interpreter Cohort Table

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_previouslivebirths')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (maternity_interpreter_cohort
.select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'person_id_demo', 'Dob', 'eth5', 'region', 'imd_quintile', 'imd_decile', 'in_gdppr', 'gdppr_min_date', 'interpreter_use', 'record_before_lookback', 'ageatbookingmother', 'delivery_date', 'agefinal', 'folicacid', 'ovsvischcat', 'ovsvischcatappdate', 'complexsocialfactors', 'gestagebooking', 'previouslivebirths')
        )



# 2 Load MSDS table



In [0]:
msds_demo= (spark.table(f'{dbc_old}.msds_v2_demographics_booking_and_pregnancy_all_years_archive')
          #.filter(F.col('ADMIDATE') > "2018-01-01")
          .filter(f.col('archived_on') == tmp_archived_on)
          )

In [0]:
msds_demo.count()

In [0]:
msds_demo= (
    msds_demo
    .select(f.col('person_id_mother_deid').alias('person_id_mother_msds'), 
            f.col('uniqpregid').alias('uniquepregid_msds'),
           # f.col('ovsvischcat').alias('ovsvischcat'),
           # f.col('ovsvischcatappdate').alias('ovsvischcatappdate'),
           #f.col('langcode').alias('langcode'),
           #f.col('complexsocialfactorsind').alias('complexsocialfactors'),
           #f.col('previouslivebirths').alias('previouslivebirths'),
           f.col('previousstillbirths').alias('previousstillbirths'),
           #f.col('previouslosseslessthan24weeks').alias('previouslosseslessthan24weeks'),
          #f.col('folicacidsupplement').alias('folicacid1'),
          #f.col('gestagebooking').alias('gestagebooking'),
          #f.col('archived_on').alias('archived_on'),
          #           .distinct() 
           ).dropDuplicates()
)

In [0]:
msds_demo.count()

In [0]:
display(msds_demo)

In [0]:
msds_demo_previousstillbirths =    (
msds_demo
.select(f.col('person_id_mother_msds').alias('person_id_mother_msds'), 
            f.col('uniquepregid_msds').alias('uniquepregid_msds'),
            #f.col('archived_on').alias('archived_on'),
          f.col('previousstillbirths').alias('previousstillbirths')))



#3 Clean previous live births variable

In [0]:
#Code from CCU018_02_D03-cohort msds

msds_demo_previousstillbirths = (msds_demo_previousstillbirths
#        .select("uniqpregid" ,  "antenatalappdate"  , "previouscaesareansections")
        .where(f.col("previousstillbirths").isNotNull())
        .dropDuplicates(["uniquepregid_msds", "previousstillbirths" ])
        # dropping anomalous large numbers
        .where( ( f.col("previousstillbirths") >= min_stillbirths ) & ( f.col("previousstillbirths") <= max_stillbirths ) )
        # logic model such that highest number in pregnancy is prioritised
        .sort("uniquepregid_msds", "previousstillbirths", ascending= False)
        .dropDuplicates(["uniquepregid_msds"])
)


#if checks_on:
display(msds_demo_previousstillbirths)



In [0]:
tab(msds_demo_previousstillbirths, 'previousstillbirths')

In [0]:
##check
count_var(msds_demo_previousstillbirths, 'person_id_mother_msds')

In [0]:
##check
count_var(msds_demo_previousstillbirths, 'uniquepregid_msds')

In [0]:
display(msds_demo_previousstillbirths)

#4 Join MSDS to Maternity interpreter cohort table 

In [0]:
##Attempting left join based on https://www.geeksforgeeks.org/pyspark-join-types-join-two-dataframes/

##check row count in each table before and after join (row count in output table will be same as left table row count - filtered lookup)

# left join on two dataframes 
maternity_interpreter_previousstillbirths=maternity_interpreter_cohort.join(msds_demo_previousstillbirths, 
               maternity_interpreter_cohort.uniqpregid == msds_demo_previousstillbirths.uniquepregid_msds,  
               "left")




#display table
display(maternity_interpreter_previousstillbirths)
display(maternity_interpreter_previousstillbirths.printSchema())

In [0]:
count_var(maternity_interpreter_previousstillbirths, 'person_id_mother_deid')

#5 Save table with previousstillbirths

In [0]:
outName = f'{proj}_maternity_interpreter_previousstillbirths'

# save
maternity_interpreter_previousstillbirths.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
maternity_interpreter_previousstillbirths = spark.table(f'{dbc}.{proj}_maternity_interpreter_previousstillbirths')